# sklearn-LDA

代码示例：https://mp.weixin.qq.com/s/hMcJtB3Lss1NBalXRTGZlQ （玉树芝兰） <br>
可视化：https://blog.csdn.net/qq_39496504/article/details/107125284  <br>
sklearn lda参数解读:https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html
<br>中文版参数解读：https://blog.csdn.net/TiffanyRabbit/article/details/76445909
<br>LDA原理-视频版：https://www.bilibili.com/video/BV1t54y127U8
<br>LDA原理-文字版：https://www.jianshu.com/p/5c510694c07e
<br>score的计算方法：https://github.com/scikit-learn/scikit-learn/blob/844b4be24d20fc42cc13b957374c718956a0db39/sklearn/decomposition/_lda.py#L729
<br>主题困惑度1：https://blog.csdn.net/weixin_43343486/article/details/109255165
<br>主题困惑度2：https://blog.csdn.net/weixin_39676021/article/details/112187210

## 1.预处理

In [1]:
import os
import pandas as pd
import re
import jieba
import jieba.posseg as psg

In [2]:
output_path = 'D:\\pythonProject5\\3_scikit learn\\C06_lda\\result'
file_path = 'D:\\pythonProject5\\3_scikit learn\\C06_lda\\data'
os.chdir(file_path)
data=pd.read_excel("data.xlsx")#content type
os.chdir(output_path)
dic_file = "D:\\pythonProject5\\3_scikit learn\\C06_lda\\stop_dic\\dict.txt"
stop_file = "D:\\pythonProject5\\3_scikit learn\\C06_lda\\stop_dic\\stopwords.txt"

In [3]:
def chinese_word_cut(mytext):
    jieba.load_userdict(dic_file)
    jieba.initialize()
    try:
        stopword_list = open(stop_file,encoding ='utf-8')
    except:
        stopword_list = []
        print("error in stop_file")
    stop_list = []
    flag_list = ['n','nz','vn']
    for line in stopword_list:
        line = re.sub(u'\n|\\r', '', line)
        stop_list.append(line)
    
    word_list = []
    #jieba分词
    seg_list = psg.cut(mytext)
    for seg_word in seg_list:
        word = re.sub(u'[^\u4e00-\u9fa5]','',seg_word.word)
        #word = seg_word.word  #如果想要分析英语文本，注释这行代码，启动下行代码
        find = 0
        for stop_word in stop_list:
            if stop_word == word or len(word)<2:     #this word is stopword
                    find = 1
                    break
        if find == 0 and seg_word.flag in flag_list:
            word_list.append(word)      
    return (" ").join(word_list)

In [4]:
print(data.content[0])
for i in range(len(data.content)):
    if type(data.content[i]) == float or type(data.content[i]) == int:
        print(i)
        # data.content[i] = None
        # data.loc['content', i] = None
        data.drop(i, inplace=True)

print("="*200)
for i in range(len(data.content)):
    if type(data.content[i]) == float:
        print(i)
     

  
250
282
314
345
375
406
437
465
483
498
539
575
595
612
631
652
702
728
742
778
782
854
880
913
917
945
947
955
995
1025
1056
1078
1107
1119
1155
1174
1177
1181
1196
1197
1198
1199
1200
1202
1203
1205
1208
1209
1210
1212
1214
1216
1218
1221
1226
1229
1231
1232
1233
1235
1237
1239
1241
1242
1248
1299
1335
1338
1375
1416
1449
1451
1453
1487
1519
1522
1525
1536
1578
1606
1628
1644
1651
1688
1722
1730
1771
1804
1841
1880
1938
1967
1972
1979
2044
2077
2092
2098
2135
2157
2200
2233
2272
2303
2327
2329
2337
2353
2391
2430
2466
2470
2494
2499
2533
2576
2608
2629
2665
2708
2745
2785
2823
2859
2899
2935
2965
2971
2983
3025
3064
3102
3132
3161
3199
3232
3259
3285
3287
3290
3310
3311
3312
3313
3314
3315
3335
3389
3402
3441
3485
3512
3525
3551
3584
3616
3620
3636
3639
3647
3649
3662
3681
3688
3690
3698
3713
3745
3762
3772
3798
3822
3853


KeyError: 250

In [ ]:
data["content"] = data.content.apply(chinese_word_cut)

## 2.LDA分析

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [ ]:
def print_top_words(model, feature_names, n_top_words):
    tword = []
    for topic_idx, topic in enumerate(model.components_):
        print("Topic #%d:" % topic_idx)
        topic_w = " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
        tword.append(topic_w)
        print(topic_w)
    return tword

In [ ]:
n_features = 1000 #提取1000个特征词语
tf_vectorizer = CountVectorizer(strip_accents = 'unicode',
                                max_features=n_features,
                                stop_words='english',
                                max_df = 0.5,
                                min_df = 1) # 原来是0.5 10
tf = tf_vectorizer.fit_transform(data.content_cutted_out)

In [ ]:
n_topics = 7
lda = LatentDirichletAllocation(n_components=n_topics, max_iter=50,
                                learning_method='batch',
                                learning_offset=50,
#                                 doc_topic_prior=0.1,
#                                 topic_word_prior=0.01,
                               random_state=0)
lda.fit(tf)

### 2.1输出每个主题对应词语 

In [ ]:
n_top_words = 10
tf_feature_names = tf_vectorizer.get_feature_names_out()
# tf_feature_names = tf_vectorizer.get_feature_names()
topic_word = print_top_words(lda, tf_feature_names, n_top_words)

### 2.2输出每篇文章对应主题 

In [ ]:
import numpy as np

In [ ]:
topics=lda.transform(tf)

In [ ]:
topic = []
for t in topics:
    topic.append("Topic #"+str(list(t).index(np.max(t))))
data['概率最大的主题序号']=topic
data['每个主题对应概率']=list(topics)
data.to_excel("data_topic.xlsx",index=False)

### 2.3可视化 

In [ ]:
import pyLDAvis
import pyLDAvis.lda_model
# import pyLDAvis.sklearn

In [ ]:
# 计算文档长度和词频
doc_length = tf.sum(axis=1).getA1()  # 获取每个文档的总词数
term_frequency = tf.sum(axis=0).getA1()  # 获取每个词的全局词频
pyLDAvis.enable_notebook()
panel = pyLDAvis.prepare(
    topic_term_dists=lda.components_,
    doc_topic_dists=lda.transform(tf),
    doc_lengths=doc_length,
    vocab=tf_vectorizer.get_feature_names_out(),
    term_frequency=term_frequency
)

pyLDAvis.display(panel)
pyLDAvis.save_html(panel, 'lda_pass'+str(n_topics)+'.html')
# 
# pyLDAvis.enable_notebook()
# pic = pyLDAvis.sklearn.prepare(lda, tf, tf_vectorizer)
## pyLDAvis.display(pic)
# pyLDAvis.save_html(pic, 'lda_pass'+str(n_topics)+'.html')
## pyLDAvis.display(pic)
#去工作路径下找保存好的html文件
#和视频里讲的不一样，目前这个代码不需要手动中断运行，可以快速出结果

### 2.4困惑度 

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plexs = []
scores = []
n_max_topics = 25
for i in range(1,n_max_topics):
    print(i)
    lda = LatentDirichletAllocation(n_components=i, max_iter=50,
                                    learning_method='batch',
                                    learning_offset=50,random_state=0)
    lda.fit(tf)
    plexs.append(lda.perplexity(tf))
    scores.append(lda.score(tf))

In [ ]:
n_t=24#区间最右侧的值。注意：不能大于n_max_topics
x=list(range(1,n_t+1))
plt.plot(x,plexs[0:n_t])
plt.xlabel("number of topics")
plt.ylabel("perplexity")
plt.show()